# Apache Spark — Iris Flower Classification

**Course:** Selected Topics in Information Systems — Lecture 9 (Apache Spark)  
**Faculty of Computers and Information, Mansoura University**

This notebook follows the exact code style shown in Lecture 9 (slides *Simple PySpark app.*) and then extends it to the assignment requirement: bring **sample data**, choose a **classification algorithm**, and apply it to that data using Apache Spark.

**Sample data:** Iris flower dataset (3 classes — *setosa*, *versicolor*, *virginica*; 4 numeric features).  
**Classification algorithms used:**
1. Logistic Regression (`pyspark.ml.classification.LogisticRegression`)
2. Random Forest Classifier (`pyspark.ml.classification.RandomForestClassifier`)

Pipeline steps:
1. Create `SparkSession` (same boilerplate as the lecture).
2. Sanity-check Spark by running the lecture's `parallelize` / `count` example.
3. Load the Iris CSV into a Spark DataFrame.
4. Convert features into a single vector (`VectorAssembler`) and index the string label (`StringIndexer`).
5. Split the data into train / test sets (80 / 20).
6. Train each classifier and evaluate accuracy + F1 on the test set.
7. Compare the two models.

## 1. Imports and SparkSession (same as Lecture 9)

In [1]:
from pyspark.sql import SparkSession

# Same boilerplate shown in Lecture 9 (slide "Simple PySpark app.")
spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("SparkByExamples.com")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(spark)

26/05/01 22:44:35 WARN Utils: Your hostname, devin-box resolves to a loopback address: 127.0.0.1; using 172.16.4.2 instead (on interface eth0)
26/05/01 22:44:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/01 22:44:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 1.1 Quick sanity check (the exact RDD example from the lecture)

In [2]:
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])
print("RDD count =", rdd.count())

RDD count = 5


## 2. Load the Iris sample dataset

We download the classic Iris CSV from the UCI repository and read it into a Spark DataFrame. The four numeric features are the sepal/petal length and width; the label is the flower species.

In [3]:
import os
import urllib.request

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
csv_path = os.path.join(DATA_DIR, "iris.csv")

if not os.path.exists(csv_path):
    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
    )
    raw = urllib.request.urlopen(url).read().decode("utf-8").strip().splitlines()
    header = "sepal_length,sepal_width,petal_length,petal_width,species"
    with open(csv_path, "w", encoding="utf-8") as f:
        f.write(header + "\n")
        for line in raw:
            if line.strip():
                f.write(line + "\n")

print("Saved:", csv_path, "(", os.path.getsize(csv_path), "bytes )")

Saved: data/iris.csv ( 4608 bytes )


In [4]:
from pyspark.sql.types import (
    StructType,
    StructField,
    DoubleType,
    StringType,
)

schema = StructType([
    StructField("sepal_length", DoubleType(), True),
    StructField("sepal_width",  DoubleType(), True),
    StructField("petal_length", DoubleType(), True),
    StructField("petal_width",  DoubleType(), True),
    StructField("species",      StringType(), True),
])

iris = spark.read.csv(csv_path, header=True, schema=schema)
iris.printSchema()
iris.show(5)
print("Total rows:", iris.count())

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)



+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|    species|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows



Total rows: 150


### 2.1 Class distribution

In [5]:
iris.groupBy("species").count().orderBy("species").show()

+---------------+-----+
|        species|count|
+---------------+-----+
|    Iris-setosa|   50|
|Iris-versicolor|   50|
| Iris-virginica|   50|
+---------------+-----+



## 3. Feature engineering

* `VectorAssembler` packs the four numeric columns into a single `features` vector.
* `StringIndexer` converts the string `species` column into a numeric `label` column (required by Spark MLlib classifiers).

In [6]:
from pyspark.ml.feature import VectorAssembler, StringIndexer

feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
indexer   = StringIndexer(inputCol="species", outputCol="label")

data = assembler.transform(iris)
data = indexer.fit(data).transform(data)
data = data.select("features", "label", "species")
data.show(5, truncate=False)

+-----------------+-----+-----------+
|features         |label|species    |
+-----------------+-----+-----------+
|[5.1,3.5,1.4,0.2]|0.0  |Iris-setosa|
|[4.9,3.0,1.4,0.2]|0.0  |Iris-setosa|
|[4.7,3.2,1.3,0.2]|0.0  |Iris-setosa|
|[4.6,3.1,1.5,0.2]|0.0  |Iris-setosa|
|[5.0,3.6,1.4,0.2]|0.0  |Iris-setosa|
+-----------------+-----+-----------+
only showing top 5 rows



## 4. Train / test split (80 / 20)

In [7]:
train, test = data.randomSplit([0.8, 0.2], seed=42)
print("Training rows:", train.count())
print("Test rows    :", test.count())

Training rows: 126


Test rows    : 24


## 5. Model 1 — Logistic Regression

Multinomial logistic regression is a natural baseline for the 3-class Iris problem.

In [8]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    family="multinomial",
)
lr_model = lr.fit(train)
lr_pred  = lr_model.transform(test)
lr_pred.select("species", "label", "prediction", "probability").show(5, truncate=False)

26/05/01 22:44:49 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


+-----------+-----+----------+---------------------------------------------------+
|species    |label|prediction|probability                                        |
+-----------+-----+----------+---------------------------------------------------+
|Iris-setosa|0.0  |0.0       |[1.0,1.0327870156782226E-29,1.1139104985459525E-52]|
|Iris-setosa|0.0  |0.0       |[1.0,2.3934963363913998E-32,1.0322754686274273E-55]|
|Iris-setosa|0.0  |0.0       |[1.0,2.8762437010260746E-43,2.4735328260435164E-69]|
|Iris-setosa|0.0  |0.0       |[1.0,5.683945971330175E-28,1.8021442888780556E-50] |
|Iris-setosa|0.0  |0.0       |[1.0,2.224476794727927E-29,4.541596687215445E-53]  |
+-----------+-----+----------+---------------------------------------------------+
only showing top 5 rows



In [9]:
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy",
)
f1_eval  = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1",
)

lr_acc = acc_eval.evaluate(lr_pred)
lr_f1  = f1_eval.evaluate(lr_pred)
print(f"Logistic Regression  -> accuracy = {lr_acc:.4f}, F1 = {lr_f1:.4f}")

Logistic Regression  -> accuracy = 1.0000, F1 = 1.0000


## 6. Model 2 — Random Forest Classifier

An ensemble of decision trees that usually performs very well on small tabular datasets.

In [10]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=5,
    seed=42,
)
rf_model = rf.fit(train)
rf_pred  = rf_model.transform(test)
rf_pred.select("species", "label", "prediction").show(5, truncate=False)

rf_acc = acc_eval.evaluate(rf_pred)
rf_f1  = f1_eval.evaluate(rf_pred)
print(f"Random Forest        -> accuracy = {rf_acc:.4f}, F1 = {rf_f1:.4f}")

+-----------+-----+----------+
|species    |label|prediction|
+-----------+-----+----------+
|Iris-setosa|0.0  |0.0       |
|Iris-setosa|0.0  |0.0       |
|Iris-setosa|0.0  |0.0       |
|Iris-setosa|0.0  |0.0       |
|Iris-setosa|0.0  |0.0       |
+-----------+-----+----------+
only showing top 5 rows



Random Forest        -> accuracy = 1.0000, F1 = 1.0000


### 6.1 Feature importances (Random Forest)

In [11]:
importances = list(zip(feature_cols, rf_model.featureImportances.toArray()))
importances.sort(key=lambda kv: kv[1], reverse=True)
for name, score in importances:
    print(f"  {name:<14s} {score:.4f}")

  petal_width    0.4718
  petal_length   0.4057
  sepal_length   0.0986
  sepal_width    0.0238


## 7. Comparison

In [12]:
from pyspark.sql import Row

results = spark.createDataFrame([
    Row(model="LogisticRegression", accuracy=float(lr_acc), f1=float(lr_f1)),
    Row(model="RandomForest",       accuracy=float(rf_acc), f1=float(rf_f1)),
])
results.show(truncate=False)

+------------------+--------+---+
|model             |accuracy|f1 |
+------------------+--------+---+
|LogisticRegression|1.0     |1.0|
|RandomForest      |1.0     |1.0|
+------------------+--------+---+



## 8. Stop the SparkSession

In [13]:
spark.stop()

---

**Summary**

* Used the same `SparkSession` boilerplate from Lecture 9.
* Loaded the Iris **sample data** (150 rows, 4 features, 3 classes).
* Built a Spark MLlib pipeline (`VectorAssembler` + `StringIndexer`).
* Trained two **classification algorithms** (Logistic Regression and Random Forest) and evaluated them with `MulticlassClassificationEvaluator`.
* Both models achieve high accuracy on the held-out test set, fulfilling the assignment requirement to *"bring sample data and choose a classification algorithm and apply it to that data"* using Apache Spark.